# Alloy blending problem

The company Steelco has received an order for 500 tonnes of steel to be used in shipbuilding.  The steel must have the following characteristics:

| Chemical Element | Minimum Grade (%) | Maximum Grade (%) |
|------------------|---------------|---------------|
| Carbon (C)       | 2             | 3             |
| Copper (Cu)      | 0.4           | 0.6           |
| Manganese (Mn)   | 1.2           | 1.65          |

The company has seven different raw materials in stock that may be used for the production of this steel. The following table lists the grades, available amounts and prices for all materials:

| Raw Material | C%     | Cu%    | Mn%    | Availability in t | Cost in dollars/t  |
|--------------|--------|--------|--------|-------------------|--------------|
| Iron1        | 2.5    | 0      | 1.3    | 400               | 200          |
| Iron2        | 3      | 0      | 0.8    | 300               | 250          |
| Iron3        | 0      | 0.3    | 0      | 600               | 150          |
| Cu1          | 0      | 90     | 0      | 500               | 220          |
| Cu2          | 0      | 96     | 4      | 200               | 240          |
| Al1          | 0      | 0.4    | 1.2    | 300               | 200          |
| Al2          | 0      | 0.6    | 0      | 250               | 165          |

The objective is to determine the composition of the steel that minimizes the production cost.

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. JuMP, HiGHS, NamedArrays, and Printf are already included. No external data files are needed.

The model allows fractional tonnes and assumes that the total mass of the steel equals the total mass of the raw materials used.

## Problem data

Rows of `composition` correspond to raw materials and columns correspond to chemical elements. The entries are percentages: for example, `2.5` means 2.5%, not a fraction of 2.5.

The `NamedArray` named `α` lets us select a composition by labels, such as `α[:iron1, :C]`. Availability and cost are in tonnes and dollars per tonne; `minimum_grade` and `maximum_grade` use the same percentage scale as `α`.

In [ ]:
using JuMP, HiGHS, NamedArrays, Printf
import MathOptInterface as MOI

raw = [:iron1, :iron2, :iron3, :cu1, :cu2, :al1, :al2]
elements = [:C, :Cu, :Mn]

# Rows are raw materials; columns are elements, in the orders above.
# Composition values are percentages.
composition = [
    2.5 0 1.3
    3 0 0.8
    0 0.3 0
    0 90 0
    0 96 4
    0 0.4 1.2
    0 0.6 0
]
α = NamedArray(composition, (raw, elements), ("Raw Material", "Element"))

# Availability in tonnes.
availability = Dict(
    :iron1 => 400,
    :iron2 => 300,
    :iron3 => 600,
    :cu1 => 500,
    :cu2 => 200,
    :al1 => 300,
    :al2 => 250,
)

# Dict(zip()) pairs each label with its corresponding value.
cost = Dict(zip(raw, [200, 250, 150, 220, 240, 200, 165]))
minimum_grade = Dict(zip(elements, [2, 0.4, 1.2]))
maximum_grade = Dict(zip(elements, [3, 0.6, 1.65]))
demand = 500
α

## Formulate the blending LP

Let $x_r$ be the tonnes of raw material $r$ used and let $p = \sum_r x_r$ be total production. The cost per tonne is $c_r$, the available stock is $u_r$, and demand is $d = 500$ tonnes.

For element $e$, let $\alpha_{re}$ be its percentage in raw material $r$. Its percentage in the finished steel is the weighted average

$$
\frac{\sum_r \alpha_{re} x_r}{p}.
$$

Production is positive because $p \geq 500$, so we can multiply the grade bounds by $p$. If $L_e$ and $U_e$ are the minimum and maximum percentages, the model is

$$
\begin{aligned}
\min_x \quad & \sum_r c_r x_r \\
\text{subject to}\quad
& 0 \leq x_r \leq u_r && \forall r, \\
& p = \sum_r x_r \geq d, \\
& \sum_r \alpha_{re} x_r \geq L_e p && \forall e, \\
& \sum_r \alpha_{re} x_r \leq U_e p && \forall e.
\end{aligned}
$$

Every expression is linear in $x$: $p$ is the sum of the material amounts, and the grade limits are constants. Since both composition and grade limits are expressed as percentages, the factors of $1/100$ cancel on the two sides of each grade constraint.

The minimum constraints use `minimum_grade`, and the maximum constraints use `maximum_grade`. Using the minimum in both would force the alloy to equal the minimum grade for every element.

We retain the original requirement to produce **at least** 500 tonnes. With these positive costs, any feasible blend above 500 tonnes could be scaled down to 500, preserving its percentages and respecting availability while reducing cost. Thus an optimal blend produces exactly 500 tonnes.

In [ ]:
model = Model(HiGHS.Optimizer)
set_silent(model)  # Remove this line to see the solver log.

@variable(model, x[raw] >= 0)  # Tonnes of each raw material.
@objective(model, Min, sum(cost[r] * x[r] for r in raw))

# An expression reuses the sum without introducing a new decision variable.
@expression(model, production, sum(x[r] for r in raw))
@constraint(model, avail[r in raw], x[r] <= availability[r])
@constraint(model, min_grade[e in elements],
    sum(α[r, e] * x[r] for r in raw) >= minimum_grade[e] * production)
@constraint(model, max_grade[e in elements],
    sum(α[r, e] * x[r] for r in raw) <= maximum_grade[e] * production)
@constraint(model, meet_demand, production >= demand)

model

## Solve, check, and report

Check that HiGHS found an optimum and that a feasible solution is available before reading any values. The report follows the raw-material order and omits amounts smaller than `1e-6` tonnes.

In [ ]:
optimize!(model)
status = termination_status(model)
println("Termination status: ", status)
status == MOI.OPTIMAL || error("HiGHS stopped with status $(status).")
is_solved_and_feasible(model) || error("No feasible optimal solution is available.")

minimum_cost = objective_value(model)
total_production = value(production)
solution = Dict(r => value(x[r]) for r in raw if value(x[r]) > 1e-6)

@printf("\nMinimum production cost: \$%.2f\n", minimum_cost)
@printf("Total steel produced: %.2f tonnes\n", total_production)
@printf("Cost per tonne: \$%.2f\n", minimum_cost / total_production)
for r in raw
    if haskey(solution, r)
        @printf("Use %.4f tonnes of %s\n", solution[r], r)
    end
end

## Check the finished alloy and remaining stock

Use the unrounded solution to compute each element's percentage in the blend. Compare it with both grade limits, then check how much of each raw material remains.

The displayed amounts are rounded for readability. Rebuilding a blend from rounded amounts can violate a grade limit, even when the solver's unrounded blend is feasible.

In [ ]:
final_grade = Dict(
    e => sum(α[r, e] * value(x[r]) for r in raw) / total_production
    for e in elements
)
remaining_stock = Dict(r => availability[r] - value(x[r]) for r in raw)

println("Finished alloy grades (percent):")
for e in elements
    @printf("%s: %.4f%% (minimum %.2f%%, maximum %.2f%%)\n",
        e, final_grade[e], minimum_grade[e], maximum_grade[e])
end

println("\nRaw material use and remaining stock (tonnes):")
for r in raw
    @printf("%s: used %.4f, available %.2f, remaining %.4f\n",
        r, value(x[r]), availability[r], remaining_stock[r])
end

## Interpret the solution

The minimum cost is approximately **98,121.64 dollars** for 500 tonnes of steel.

Which elements are at a minimum or maximum grade? Which raw-material availability limits are binding? How do those limits explain why buying only the cheapest material cannot satisfy the order?

Compare this model with the diet LPs: the alloy constraints bound the **composition of the mixture**, so their right-hand sides depend on total production. They also impose both lower and upper grade limits.